# 03 · Coulson ingests
*Files arrive late, twice, and in new shapes. Auto Loader does not mind.*

Agent Coulson's field teams deliver reports whenever they can. Your job is to load each delivery into **bronze** Delta tables exactly once, no matter how often you run the job, and to keep going when a delivery brings a new field.
You will simulate the deliveries yourself, so you can watch each effect. **Expect one deliberate failure** in Step 5, followed by a successful restart in Step 6.

| Concept | Where you meet it |
|---|---|
| Landing zone, bronze layer and lineage columns | the first explainer below, Step 1 |
| Auto Loader: loads only new files, remembers them in a checkpoint | the first explainer, Steps 1 to 4 |
| `trigger(availableNow=True)`: incremental batch | the first explainer, Step 3 |
| Schema drift and schema evolution (`addNewColumns`) | the second explainer, Steps 5 and 6 |
| Bronze keeps duplicates and bad rows | Step 7 |

Deep dive: [docs/03-coulson-ingests.md](../../docs/03-coulson-ingests.md)

**Needs:**
* Chapter 01 ran (the schema `shield.bronze` and the volumes `staging`, `raw`, `checkpoints`).
* Chapter 02 ran, Step 1 (it generates the files in `shield.landing.staging` that this notebook delivers). If you skipped it, the setup cell tells you; run the `generate(...)` call from chapter 02 Step 1 and come back.
* Serverless compute (or a cluster) and the right to create tables in `shield.bronze`.

**Running it twice:** every cell is safe to repeat, but the numbers of Steps 1 to 6 describe a **fresh start**. A second run finds everything already delivered and loaded, so the counts that grow in a fresh run stay at 10,250 and some checks show ❌. That is correct. To start over, use the reset cell under Clean up.
The **Expected** numbers assume the dataset of chapter 02 (10,000 incidents, seed 42, 5 % dirty, 5 files); other data gives other numbers. They are expectations until you have run the notebook in a workspace.

## Auto Loader: incremental ingestion into bronze

**The idea:** a *landing zone* is a folder where files are put exactly as delivered, and never edited. **Auto Loader** is a Databricks feature that watches such a folder, loads only the files it has not seen yet, and remembers what it has done in a **checkpoint**. **Bronze** is the first layer of the medallion pattern (chapter 05 explains all three layers): the raw data as received, plus columns that say where each row came from.

```
 staging volume            raw volume (landing zone)        Auto Loader ("cloudFiles")              bronze table
 all generated files ─deliver()─▶ files that have     ──▶ reads only NEW files           ──▶ shield.bronze.<feed>
 (chapter 02)                     "arrived" so far        remembers them in the checkpoint        + _source_file, _ingested_at
```

**How this notebook implements it**

| What | Path or name |
|---|---|
| All generated files, the "field" | `/Volumes/shield/landing/staging/<feed>` |
| Landing zone: a file is copied here when it "arrives" (`deliver()`) | `/Volumes/shield/landing/raw/<feed>` |
| Which files are done | `/Volumes/shield/landing/checkpoints/<feed>/checkpoint` |
| Schema Auto Loader inferred | `/Volumes/shield/landing/checkpoints/<feed>/schema` |
| Target | `shield.bronze.<feed>` for `agents`, `locations`, `threat_assessments`, `incident_reports` |

* `ingest()` reads the landing zone with the source `cloudFiles`, adds `_source_file` (the file a row came from) and `_ingested_at` (when), and writes to the bronze table. Auto Loader adds one more column, `_rescued_data`, which collects values that do not fit the schema. Nothing is cleaned.
* Auto Loader is a *streaming* source, and a stream normally runs forever. `trigger(availableNow=True)` says "process everything that is new, then stop", so each call to `ingest()` is an incremental **batch**. This is the trigger that serverless compute supports; the default (a stream that never stops) is rejected there.
* `toTable(...)` starts the stream and returns at once, so `ingest()` calls `awaitTermination()` to wait until it is done. That also makes a failed stream raise its error in the cell that started it.
* `inferColumnTypes` makes Auto Loader guess real types (numbers, booleans). Without it, JSON and CSV columns are all strings.

**Why it helps:** a re-run costs almost nothing because old files are skipped, and if the logic downstream changes you can rebuild from the untouched files in the landing zone.

**Not covered yet:** ingesting `sensor_pings` (a mini-project in the doc, and streaming in chapter 09), scheduling the ingest (chapter 10), the same job as a declarative pipeline (chapter 08), and cleaning the data (chapters 05 and 06).

## Setup

In [ ]:
import shutil
from pathlib import Path

from pyspark.sql import functions as F

STAGING = Path("/Volumes/shield/landing/staging")
RAW = Path("/Volumes/shield/landing/raw")
CP = "/Volumes/shield/landing/checkpoints"


def show_checks(checks):
    """Show (check, result, expected) rows with a ✅ / ❌ for whether result matches expected."""
    fmt = lambda v: f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else str(v)
    rows = [(what, fmt(got), fmt(want), "✅" if got == want else "❌") for what, got, want in checks]
    display(spark.createDataFrame(rows, "check string, result string, expected string, ok string"))


def deliver(feed: str, total: int = 1) -> None:
    """Make sure the first `total` staged files of a feed are in the raw landing zone (like a field delivery).

    Files already delivered are skipped, so calling it again with the same `total` does nothing."""
    (RAW / feed).mkdir(parents=True, exist_ok=True)
    delivered = {p.name for p in (RAW / feed).glob("*")}
    todo = [p for p in sorted((STAGING / feed).glob("*"))[:total] if p.name not in delivered]
    for p in todo:
        shutil.copy(p, RAW / feed / p.name)
    print(f"{feed}: delivered {[p.name for p in todo]}")


def ingest(feed: str, fmt: str, table: str) -> None:
    """Incrementally load new files of `feed` into a bronze table with Auto Loader."""
    reader = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", fmt)
        .option("cloudFiles.schemaLocation", f"{CP}/{feed}/schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    )
    if fmt == "csv":
        reader = reader.option("header", "true")
    query = (
        reader.load(f"{RAW}/{feed}")
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
        .writeStream.option("checkpointLocation", f"{CP}/{feed}/checkpoint")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)  # process what is there, then stop
        .toTable(f"shield.bronze.{table}")
    )
    query.awaitTermination()  # toTable returns at once; wait, and raise the stream's error here


def bronze(table: str):
    return spark.table(f"shield.bronze.{table}")


schemas = {row[0] for row in spark.sql("SHOW SCHEMAS IN shield").collect()}
assert "bronze" in schemas, "Schema shield.bronze not found. Run notebook 01 first."
for feed in ["agents", "locations", "threat_assessments", "incident_reports"]:
    assert any((STAGING / feed).glob("*")), f"No staged files in {STAGING / feed}. Run notebook 02, Step 1 first."

## 1 · Dimension feeds

### Step 1 · Three CSV feeds, one delivery each
**Goal:** load the small reference feeds (agents, locations, threat assessments) into bronze with Auto Loader.

| Input | Value |
|---|---|
| `staging/agents/agents.csv` | 50 agents |
| `staging/locations/locations.csv` | 10 locations |
| `staging/threat_assessments/threat_assessments.csv` | 120 assessments (30 subjects, 4 versions each) |

**Expected:** three bronze tables with 50, 10 and 120 rows. Each has the lineage columns `_source_file` and `_ingested_at`. The first run of a stream can take a minute on serverless.

What you will see:

| check | result | expected | ok |
|---|---|---|---|
| shield.bronze.agents rows | 50 | 50 | ✅ |
| shield.bronze.locations rows | 10 | 10 | ✅ |
| shield.bronze.threat_assessments rows | 120 | 120 | ✅ |
| lineage columns on all three | True | True | ✅ |

Safe to re-run: the file is already delivered and already loaded, so nothing happens and the counts stay.

In [ ]:
dimension_counts = {"agents": 50, "locations": 10, "threat_assessments": 120}
for feed in dimension_counts:
    deliver(feed)
    ingest(feed, "csv", feed)

lineage_columns = {"_source_file", "_ingested_at"}
show_checks(
    [(f"shield.bronze.{feed} rows", bronze(feed).count(), want) for feed, want in dimension_counts.items()]
    + [("lineage columns on all three", all(lineage_columns <= set(bronze(feed).columns) for feed in dimension_counts), True)]
)

## 2 · Incident reports trickle in

The incident feed was generated as five files of 2,050 rows each (10,250 rows in total, including 250 replayed duplicates). You deliver them in three rounds: the first file, files 2 and 3, then files 4 and 5.

### Step 2 · First delivery
**Goal:** create the incident bronze table from the first file. This cell creates the state the next steps build on.

**Expected:** `shield.bronze.incident_reports` has 2,050 rows from one source file. The table has no `source_channel` column yet.

Safe to re-run: `deliver(..., total=1)` means "the first file is there", it never delivers more. After later steps the table is bigger than 2,050 rows, so the row check then shows ❌ (see "Running it twice" at the top).

In [ ]:
deliver("incident_reports", total=1)
ingest("incident_reports", "json", "incident_reports")

bronze_incidents = bronze("incident_reports")
show_checks([
    ("rows", bronze_incidents.count(), 2_050),
    ("source files", bronze_incidents.select("_source_file").distinct().count(), 1),
    ("has a source_channel column", "source_channel" in bronze_incidents.columns, False),
])

### Step 3 · Run again with nothing new
**Goal:** show what the checkpoint does. The same call, with no new file in the landing zone, must load nothing.

**Expected:** the row count is exactly the same as after Step 2 (2,050), because Auto Loader remembers the file it already loaded.

Safe to re-run: that is the whole point.

In [ ]:
rows_before = bronze("incident_reports").count()
ingest("incident_reports", "json", "incident_reports")
rows_after = bronze("incident_reports").count()

show_checks([
    ("rows before the second run", rows_before, 2_050),
    ("rows after the second run", rows_after, 2_050),
    ("unchanged", rows_after == rows_before, True),
])

### Step 4 · Two more files
**Goal:** show that new files are picked up and old ones are not read again.

**Expected:** the second and third file (`part-0001.json` and `part-0002.json`) are delivered and the table grows to 6,150 rows from three source files. These files still have the original fields, so there is still no `source_channel` column.

Safe to re-run: `total=3` means "the first three files are there", so a second run delivers and loads nothing. After Step 6 the row checks show ❌ because the table has grown.

In [ ]:
deliver("incident_reports", total=3)
ingest("incident_reports", "json", "incident_reports")

bronze_incidents = bronze("incident_reports")
show_checks([
    ("rows", bronze_incidents.count(), 6_150),
    ("source files", bronze_incidents.select("_source_file").distinct().count(), 3),
    ("has a source_channel column", "source_channel" in bronze_incidents.columns, False),
])

## Schema drift and schema evolution

**The idea:** a feed you do not control will change. **Schema drift** is the feed starting to send a field your table has never seen. **Schema evolution** is how your pipeline reacts. Auto Loader stores the schema it inferred, and an evolution mode says what happens when a new field shows up. This is an Auto Loader option, not a general Spark feature.

```
 files 1-3:  agent_id, casualties, incident_id, ...                   (inferred once, saved in the schema folder)
 file 4:     ... + source_channel   ← new field
                │
     run 1 ──▶  Auto Loader saves the new schema, then the stream STOPS with an error   (nothing from this run is written)
     run 2 ──▶  starts again with the saved schema, loads files 4 and 5, table gets the column
```

**How this notebook implements it**

* `cloudFiles.schemaEvolutionMode = addNewColumns` (also the default when you give no schema): the first run that meets a new field fails once, after saving the new schema. `mergeSchema` on the write lets the bronze table add the column. Existing rows get `NULL` in it.
* The generator adds `source_channel` (`radio`, `satellite` or `informant`) to the last 20 % of the incidents. That starts in the 4th file `part-0003.json`, where 200 of the 2,050 rows have it; 1,845 of the 2,050 rows of the 5th file `part-0004.json` have it too (the other 205 are replayed duplicates of older incidents, which never had it). So the drift is hit by the **fourth** file, and Step 4 stops after the third on purpose.
* Step 5 delivers files 4 and 5 and catches the expected error. Step 6 is the restart.

**The other modes:** `rescue` never fails and puts new fields into `_rescued_data`; `failOnNewColumns` fails and keeps failing until you change the schema or remove the file; `none` ignores new fields. They are mini-projects in the doc.

**Why it helps:** no data is lost and nobody edits the table by hand. In production you do not restart by hand either: a Lakeflow Job with retries does it (chapter 10).

**Not covered yet:** `rescue` and `failOnNewColumns` in practice (doc mini-project), type changes (a column that changes from number to text lands in `_rescued_data`).

## 3 · Schema drift

### Step 5 · A new field arrives (the deliberate failure)
**Goal:** see the stream fail once, and confirm that the failure loaded nothing.

| Input | Value |
|---|---|
| `part-0003.json` | 2,050 rows, 200 of them with `source_channel` |
| `part-0004.json` | 2,050 rows, 1,845 of them with `source_channel` |

**Expected:** the cell prints `Expected failure:` followed by the name and the start of the error message, an error about an unknown field. This is Auto Loader doing its job, not a bug. The error is caught here so the notebook goes on. The table should still have 6,150 rows, because the failed run does not commit the batch that found the field (this behaviour is from the Databricks documentation; I have not run it).

What you will see (the exact exception name and text are not verified):

```
incident_reports: delivered ['part-0003.json', 'part-0004.json']
Expected failure: StreamingQueryException ...unknown field... source_channel ...
```

Re-running this cell is the restart itself: the new schema is already saved, so the second run succeeds and the first check shows ❌. That is expected, and Step 6 then has nothing left to load.

In [ ]:
deliver("incident_reports", total=5)
drift_error = None
try:
    ingest("incident_reports", "json", "incident_reports")
except Exception as error:  # noqa: BLE001
    drift_error = error
    print("Expected failure:", type(error).__name__, str(error)[:200])

show_checks([
    ("the stream failed on the new field", drift_error is not None, True),
    ("rows after the failed run", bronze("incident_reports").count(), 6_150),
])

### Step 6 · Restart: the new column is there
**Goal:** run the same `ingest()` again, unchanged, and see the stream succeed with the evolved schema.

**Expected:** all 10,250 rows are loaded from five source files. The table now has a `source_channel` column. 2,045 rows have a value in it (200 + 1,845), and the 8,205 older rows have `NULL`. The table below shows 5 of the 2,045 rows (`limit(5)`); which rows you get is not guaranteed.

| incident_id | severity | source_channel | _source_file |
|---|---|---|---|
| I0008001 | low | satellite | .../raw/incident_reports/part-0003.json |

Safe to re-run: once everything is loaded, a further run loads nothing.

In [ ]:
ingest("incident_reports", "json", "incident_reports")

bronze_incidents = bronze("incident_reports")
display(
    bronze_incidents.filter("source_channel IS NOT NULL")
    .select("incident_id", "severity", "source_channel", "_source_file")
    .limit(5)
)
show_checks([
    ("rows", bronze_incidents.count(), 10_250),
    ("source files", bronze_incidents.select("_source_file").distinct().count(), 5),
    ("has a source_channel column", "source_channel" in bronze_incidents.columns, True),
    ("rows with a source_channel", bronze_incidents.filter("source_channel IS NOT NULL").count(), 2_045),
])

## 4 · What bronze holds

### Step 7 · Bronze keeps everything
**Goal:** prove that nothing was cleaned: duplicates and defective rows are all still in bronze. This is the audit trail, and chapters 05 and 06 deal with them downstream.

**Expected** (rows, duplicates included):

| check | expected |
|---|---|
| rows | 10,250 |
| distinct `incident_id` | 10,000 |
| `incident_id`s that appear twice | 250 |
| missing agent | 96 |
| severity not in low / medium / high / critical | 110 |
| rows per source file | 2,050 in each of the five files |

The table lists the rows per source file.

Safe to re-run.

In [ ]:
bronze_incidents = bronze("incident_reports")
rows_per_file = bronze_incidents.groupBy("_source_file").count().orderBy("_source_file")
display(rows_per_file)

show_checks([
    ("rows", bronze_incidents.count(), 10_250),
    ("distinct incident_id", bronze_incidents.select("incident_id").distinct().count(), 10_000),
    ("incident_ids that appear twice", bronze_incidents.groupBy("incident_id").count().filter("count > 1").count(), 250),
    ("missing agent", bronze_incidents.filter("agent_id IS NULL").count(), 96),
    ("invalid severity", bronze_incidents.filter("severity NOT IN ('low', 'medium', 'high', 'critical')").count(), 110),
    ("rows per source file", sorted(row["count"] for row in rows_per_file.collect()), [2_050] * 5),
])

## 👀 In the workspace

* **Catalog** → `shield` → `bronze`: four tables. On `incident_reports`, the **Details** tab and the **Sample data** tab show the columns, including `source_channel`, `_source_file` and `_ingested_at`.
* **Catalog** → `shield` → `landing` → `raw`, and `checkpoints`: the delivered files, and the `schema` and `checkpoint` folders that Auto Loader maintains. Do not edit these by hand.
* On the table's **History** tab you see one commit per ingest run that loaded something. Runs that loaded nothing add no row (the exact entries are not verified).

## 🧹 Clean up

Nothing is deleted: chapters 04 to 12 and 16 read the bronze tables, and chapters 08 and 10 also reuse the landing zone and the checkpoints. What this notebook created:

* tables `shield.bronze.agents`, `locations`, `threat_assessments`, `incident_reports`;
* files in `/Volumes/shield/landing/raw/<feed>` and folders `schema` and `checkpoint` in `/Volumes/shield/landing/checkpoints/<feed>`.

To start over, run the commented cell below. The three parts belong together: delete the tables, the landing files **and** the checkpoints. If you delete only one of them, Auto Loader and the table disagree about what was loaded (for example, with a deleted table and a kept checkpoint, the checkpoint still says the old files are done, so the new table never gets them). Then run this notebook from the top.

In [ ]:
# for feed in ["agents", "locations", "threat_assessments", "incident_reports"]:
#     spark.sql(f"DROP TABLE IF EXISTS shield.bronze.{feed}")
#     dbutils.fs.rm(f"{RAW}/{feed}", True)
#     dbutils.fs.rm(f"{CP}/{feed}", True)

## 🎯 Recap

| Idea | What you saw |
|---|---|
| Landing zone | `raw/<feed>` holds files exactly as delivered; nothing is edited |
| Auto Loader | Step 3: a run with no new files loads nothing; Step 4: new files are added exactly once |
| `availableNow` | each `ingest()` is an incremental batch that stops by itself |
| Schema evolution | Step 5: one expected failure on the 4th file; Step 6: a restart succeeds with the new `source_channel` column |
| Bronze | raw plus `_source_file` and `_ingested_at`; 10,250 rows including 250 duplicates and the bad rows |

Try-it-yourself mini-projects: [docs/03-coulson-ingests.md](../../docs/03-coulson-ingests.md#try-it-yourself)

> 💡 Next: chapter 04 ("Delta Lake 101") looks inside these tables: transactions, schema enforcement, `MERGE`, and history.